# Enable GPU
from the menue Runtime -> Change runtime type -> enable T4 GPU 

# Seting up the enviroment
Run the script bellow.
This will return an end point link

In [ ]:
# 1. Free port 8000 if previously occupied
!fuser -k 8000/tcp 2>/dev/null || true

# 2. Install Dependencies
!pip install -q fastapi uvicorn pydantic numpy scipy

import math, time, threading
import numpy as np
from typing import Dict, Any, Optional
from pydantic import BaseModel
from fastapi import FastAPI, Header
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title="Science Blocks DEM Solver")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def get_gpu():
    try:
        import torch
        if torch.cuda.is_available():
            return f"{torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)"
    except Exception:
        pass
    return "Cloud CPU"

class SimReq(BaseModel):
    sun_rpm: float = 200.0
    jar_rpm: float = -300.0
    r_sun: float = 1.30
    r_jar: float = 0.45
    h_jar: float = 1.10
    ball_count: int = 25
    ball_radius_m: float = 0.012
    ball_density: float = 7850.0
    frames: int = 150
    fps: float = 30.0
    substeps_per_frame: int = 40
    restitution: float = 0.82
    gravity: float = 9.81
    mill_orientation: str = "VERTICAL"

@app.get("/health")
def health(authorization: Optional[str] = Header(None)):
    return {
        "status": "online",
        "gpu": get_gpu(),
        "authorized": bool(authorization and authorization.startswith("Bearer ")),
    }

@app.post("/simulate")
def simulate(req: SimReq):
    np.random.seed(42)
    dt = (1.0 / req.fps) / req.substeps_per_frame
    w_sun = (req.sun_rpm * 2.0 * math.pi) / 60.0
    w_jar = (req.jar_rpm * 2.0 * math.pi) / 60.0
    radii = np.full(req.ball_count, req.ball_radius_m)
    masses = (4.0 / 3.0) * math.pi * (radii**3) * req.ball_density
    half_h = max(0.01, req.h_jar / 2.0 - req.ball_radius_m)
    pos = np.column_stack((
        np.random.uniform(-0.15, 0.15, req.ball_count),
        np.random.uniform(-0.15, 0.15, req.ball_count),
        np.random.uniform(-half_h * 0.5, half_h * 0.5, req.ball_count),
    ))
    vel = np.zeros_like(pos)
    traj = np.zeros((req.frames, req.ball_count, 3), dtype=np.float32)
    forces = np.zeros((req.frames, req.ball_count), dtype=np.float32)
    total_impacts = 0
    energies = []
    t = 0.0
    for f in range(req.frames):
        traj[f, :, :] = pos.astype(np.float32)
        f_frame = np.zeros(req.ball_count, dtype=np.float32)
        for _ in range(req.substeps_per_frame):
            ax = (w_sun**2) * req.r_sun * math.cos(w_jar * t) + (w_jar**2) * pos[:, 0] + 2.0 * (w_jar + w_sun) * vel[:, 1]
            ay = -(w_sun**2) * req.r_sun * math.sin(w_jar * t) + (w_jar**2) * pos[:, 1] - 2.0 * (w_jar + w_sun) * vel[:, 0]
            az = np.full(req.ball_count, -req.gravity if req.mill_orientation == "VERTICAL" else 0.0)
            acc = np.column_stack((ax, ay, az))
            pos += vel * dt + 0.5 * acc * (dt**2)
            t += dt
            r = np.sqrt(pos[:, 0]**2 + pos[:, 1]**2)
            hit = r > (req.r_jar - req.ball_radius_m)
            if np.any(hit):
                for idx in np.where(hit)[0]:
                    nx, ny = pos[idx, 0] / r[idx], pos[idx, 1] / r[idx]
                    pos[idx, 0] -= nx * (r[idx] - (req.r_jar - radii[idx]))
                    pos[idx, 1] -= ny * (r[idx] - (req.r_jar - radii[idx]))
                    vn = vel[idx, 0] * nx + vel[idx, 1] * ny
                    if vn > 0:
                        total_impacts += 1
                        fn = masses[idx] * (1.0 + req.restitution) * vn / dt
                        f_frame[idx] = max(f_frame[idx], float(fn))
                        vel[idx, 0] -= (1.0 + req.restitution) * vn * nx
                        vel[idx, 1] -= (1.0 + req.restitution) * vn * ny
                        energies.append(float(0.5 * masses[idx] * (vn**2)))
            vel += acc * dt
        forces[f, :] = f_frame
    return {
        "status": "success",
        "total_impacts": total_impacts,
        "mean_impact_energy_j": float(np.mean(energies)) if energies else 0.0,
        "max_impact_energy_j": float(np.max(energies)) if energies else 0.0,
        "trajectories": traj.tolist(),
        "velocities": vel.tolist(),
        "forces": forces.tolist(),
    }

# 3. Launch Uvicorn cleanly without signal handler conflicts
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None  # <-- Prevents Jupyter Kernel Crash!

threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print("✅ Server successfully started on port 8000!")
print(f"Hardware: {get_gpu()}")

✅ Server successfully started on port 8000!
Hardware: Cloud CPU


In [ ]:
!npx -y localtunnel --port 8000

⠙⠹⠸⠼⠴⠦your url is: https://good-clocks-drive.loca.lt
^C


In [ ]:
# Check NVIDIA GPU drivers and PyTorch CUDA state
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(f"Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected by CUDA. Please switch to T4 GPU via Runtime > Change runtime type.")

/bin/bash: line 1: nvidia-smi: command not found
PyTorch Version: 2.11.0+cpu
CUDA Available : False
⚠️ No GPU detected by CUDA. Please switch to T4 GPU via Runtime > Change runtime type.
